# Task 1 - Historical VaR (95%) & CVaR

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [5]:
nav = pd.read_csv("../data/raw/02_nav_history.csv")
nav.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [6]:
nav["date"] = pd.to_datetime(nav["date"])

nav = nav.sort_values(
    ["amfi_code", "date"]
)

nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"]
       .pct_change()
)

nav = nav.dropna()

nav.head()

,amfi_code,date,nav,daily_return
5751,100016,2022-01-04,515.0971,-0.010306
5752,100016,2022-01-05,521.7239,0.012865
5753,100016,2022-01-06,515.7880,-0.011377
5754,100016,2022-01-07,515.1639,-0.001210
5755,100016,2022-01-10,510.7136,-0.008639


In [7]:
var_report = []

for code, group in nav.groupby("amfi_code"):

    var95 = group["daily_return"].quantile(0.05)

    cvar95 = (
        group[group["daily_return"] <= var95]
        ["daily_return"]
        .mean()
    )

    var_report.append({
        "amfi_code": code,
        "VaR_95": var95,
        "CVaR_95": cvar95
    })

var_report = pd.DataFrame(var_report)

var_report.head()

,amfi_code,VaR_95,CVaR_95
0,100016,-0.014364,-0.018060
1,100025,-0.003793,-0.004994
2,100033,-0.019034,-0.023456
3,101206,-0.013282,-0.017439
4,101207,-0.026021,-0.032459


In [10]:
fund_master = pd.read_csv("../data/raw/01_fund_master.csv")

In [11]:
fund_master.head()

,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


In [12]:
var_report = var_report.merge(
    fund_master[["amfi_code", "scheme_name"]],
    on="amfi_code",
    how="left"
)

var_report.head()

,amfi_code,VaR_95,CVaR_95,scheme_name
0,100016,-0.014364,-0.018060,HDFC Top 100 Fund - Regular Plan - Growth
1,100025,-0.003793,-0.004994,HDFC Short Term Debt Fund - Regular - Growth
2,100033,-0.019034,-0.023456,HDFC Mid-Cap Opportunities Fund - Regular - Gr...
3,101206,-0.013282,-0.017439,ABSL Frontline Equity Fund - Regular - Growth
4,101207,-0.026021,-0.032459,ABSL Small Cap Fund - Regular - Growth


In [13]:
fund_master = pd.read_csv("../data/raw/01_fund_master.csv")
nav = pd.read_csv("../data/raw/02_nav_history.csv")
aum = pd.read_csv("../data/raw/03_aum_by_fund_house.csv")
sip = pd.read_csv("../data/raw/04_monthly_sip_inflows.csv")
category = pd.read_csv("../data/raw/05_category_inflows.csv")
folios = pd.read_csv("../data/raw/06_industry_folio_count.csv")
performance = pd.read_csv("../data/raw/07_scheme_performance.csv")
transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")
holdings = pd.read_csv("../data/raw/09_portfolio_holdings.csv")
benchmark = pd.read_csv("../data/raw/10_benchmark_indices.csv")

In [14]:
var_report = var_report.sort_values(
    "VaR_95"
)

var_report.head(10)

,amfi_code,VaR_95,CVaR_95,scheme_name
22,119599,-0.026859,-0.032384,SBI Small Cap Fund - Direct Plan - Growth
17,119095,-0.026188,-0.031667,Axis Small Cap Fund - Regular - Growth
4,101207,-0.026021,-0.032459,ABSL Small Cap Fund - Regular - Growth
11,118634,-0.025438,-0.032304,Nippon India Small Cap Fund - Regular - Growth
21,119598,-0.024507,-0.030595,SBI Small Cap Fund - Regular Plan - Growth
39,149324,-0.023483,-0.031036,DSP Small Cap Fund - Regular - Growth
7,102886,-0.019220,-0.023251,UTI Mid Cap Fund - Regular - Growth
2,100033,-0.019034,-0.023456,HDFC Mid-Cap Opportunities Fund - Regular - Gr...
25,120505,-0.018892,-0.024342,ICICI Pru Midcap Fund - Regular - Growth
16,119094,-0.018480,-0.024260,Axis Midcap Fund - Regular - Growth


In [15]:
var_report.to_csv(
    "../data/processed/var_cvar_report.csv",
    index=False
)

print("✅ var_cvar_report.csv saved")

✅ var_cvar_report.csv saved


task 2


In [16]:
top5 = (
    performance.sort_values("return_3yr_pct", ascending=False)
    .head(5)["amfi_code"]
    .tolist()
)

top5

[119598, 119599, 101207, 119095, 118634]

In [17]:
rolling_nav = nav[
    nav["amfi_code"].isin(top5)
].copy()

rolling_nav.head()

,amfi_code,date,nav
2300,119598,2022-01-03,89.8738
2301,119598,2022-01-04,88.5495
2302,119598,2022-01-05,88.0925
2303,119598,2022-01-06,88.5175
2304,119598,2022-01-07,91.4235


In [19]:
print(rolling_nav.columns.tolist())

['amfi_code', 'date', 'nav']


In [20]:
rolling_nav = rolling_nav.sort_values(
    ["amfi_code", "date"]
)

rolling_nav["daily_return"] = (
    rolling_nav.groupby("amfi_code")["nav"]
               .pct_change()
)

rolling_nav = rolling_nav.dropna()

rolling_nav.head()

,amfi_code,date,nav,daily_return
33351,101207,2022-01-04,38.1545,-0.010865
33352,101207,2022-01-05,38.1775,0.000603
33353,101207,2022-01-06,37.0665,-0.029101
33354,101207,2022-01-07,37.9845,0.024766
33355,101207,2022-01-10,38.0320,0.001251


In [21]:
risk_free = 0.065

rolling_nav["rolling_mean"] = (
    rolling_nav.groupby("amfi_code")["daily_return"]
               .transform(lambda x: x.rolling(90).mean())
)

rolling_nav["rolling_std"] = (
    rolling_nav.groupby("amfi_code")["daily_return"]
               .transform(lambda x: x.rolling(90).std())
)

rolling_nav["rolling_sharpe"] = (
    ((rolling_nav["rolling_mean"] * 252) - risk_free)
    /
    (rolling_nav["rolling_std"] * np.sqrt(252))
)

rolling_nav.head()

,amfi_code,date,nav,daily_return,rolling_mean,rolling_std,rolling_sharpe
33351,101207,2022-01-04,38.1545,-0.010865,NaN,NaN,NaN
33352,101207,2022-01-05,38.1775,0.000603,NaN,NaN,NaN
33353,101207,2022-01-06,37.0665,-0.029101,NaN,NaN,NaN
33354,101207,2022-01-07,37.9845,0.024766,NaN,NaN,NaN
33355,101207,2022-01-10,38.0320,0.001251,NaN,NaN,NaN


TASK 3   Investor Cohort Analysis (Single Cell)


In [22]:
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])

transactions["Year"] = transactions["transaction_date"].dt.year

first_txn = (
    transactions.groupby("investor_id")["Year"]
    .min()
    .reset_index()
    .rename(columns={"Year":"Cohort_Year"})
)

transactions = transactions.merge(
    first_txn,
    on="investor_id"
)

cohort_summary = (
    transactions.groupby("Cohort_Year")
    .agg(
        Avg_SIP=("amount_inr","mean"),
        Total_Invested=("amount_inr","sum")
    )
    .reset_index()
)

top_fund = (
    transactions.groupby(["Cohort_Year","amfi_code"])
    .size()
    .reset_index(name="Count")
)

top_fund = (
    top_fund.sort_values(
        ["Cohort_Year","Count"],
        ascending=[True,False]
    )
    .drop_duplicates("Cohort_Year")
)

top_fund = top_fund.merge(
    fund_master[
        ["amfi_code","scheme_name"]
    ],
    on="amfi_code",
    how="left"
)

cohort_summary = cohort_summary.merge(
    top_fund[
        ["Cohort_Year","scheme_name"]
    ],
    on="Cohort_Year",
    how="left"
)

cohort_summary.rename(
    columns={"scheme_name":"Top_Fund"},
    inplace=True
)

cohort_summary.to_csv(
    "../reports/investor_cohort_analysis.csv",
    index=False
)

cohort_summary

,Cohort_Year,Avg_SIP,Total_Invested,Top_Fund
0,2024,107422.541832,3491125187,Mirae Asset Emerging Bluechip Fund - Regular -...
1,2025,109158.577061,30455243,SBI Small Cap Fund - Direct Plan - Growth


Task 4 – SIP Continuity Analysis (Single Cell)

In [23]:
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])

sip = transactions[
    transactions["transaction_type"]
    .str.upper()
    .str.contains("SIP")
].copy()

sip = sip.sort_values(
    ["investor_id", "transaction_date"]
)

sip["Gap_Days"] = (
    sip.groupby("investor_id")["transaction_date"]
       .diff()
       .dt.days
)

sip_summary = (
    sip.groupby("investor_id")
       .agg(
            SIP_Count=("transaction_date", "count"),
            Avg_Gap_Days=("Gap_Days", "mean"),
            Total_Investment=("amount_inr", "sum")
       )
       .reset_index()
)

sip_summary = sip_summary[
    sip_summary["SIP_Count"] >= 6
]

sip_summary["Status"] = np.where(
    sip_summary["Avg_Gap_Days"] > 35,
    "At Risk",
    "Healthy"
)

sip_summary.to_csv(
    "../reports/sip_continuity_analysis.csv",
    index=False
)

sip_summary.head(10)

,investor_id,SIP_Count,Avg_Gap_Days,Total_Investment,Status
3,INV000004,6,85.400000,48256,At Risk
7,INV000008,6,70.400000,72853,At Risk
9,INV000010,6,64.800000,32183,At Risk
10,INV000011,7,40.166667,93920,At Risk
11,INV000012,8,57.000000,40139,At Risk
12,INV000013,7,55.333333,75613,At Risk
13,INV000014,7,75.333333,140315,At Risk
22,INV000023,8,58.571429,202529,At Risk
27,INV000028,6,93.600000,84262,At Risk
28,INV000029,7,60.666667,90012,At Risk


Task 5 – Simple Fund Recommender

In [1]:
import pandas as pd
import numpy as np

performance = pd.read_csv("../data/raw/07_scheme_performance.csv")
nav = pd.read_csv("../data/raw/02_nav_history.csv")

In [2]:
nav["date"] = pd.to_datetime(nav["date"])

nav = nav.sort_values(
    ["amfi_code", "date"]
)

nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"]
       .pct_change()
)

nav = nav.dropna()

nav.head()

,amfi_code,date,nav,daily_return
5751,100016,2022-01-04,515.0971,-0.010306
5752,100016,2022-01-05,521.7239,0.012865
5753,100016,2022-01-06,515.7880,-0.011377
5754,100016,2022-01-07,515.1639,-0.001210
5755,100016,2022-01-10,510.7136,-0.008639


In [3]:
risk_free = 0.065

sharpe_list = []

for code, group in nav.groupby("amfi_code"):

    mean_return = group["daily_return"].mean() * 252
    volatility = group["daily_return"].std() * np.sqrt(252)

    sharpe = (
        (mean_return - risk_free) / volatility
        if volatility != 0 else np.nan
    )

    sharpe_list.append({
        "amfi_code": code,
        "Sharpe_Ratio": sharpe
    })

sharpe_df = pd.DataFrame(sharpe_list)

sharpe_df.head()

,amfi_code,Sharpe_Ratio
0,100016,-0.201517
1,100025,-0.567095
2,100033,1.093699
3,101206,1.027213
4,101207,0.162661


In [4]:
recommend = performance.merge(
    sharpe_df,
    on="amfi_code",
    how="left"
)

recommend.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade,Sharpe_Ratio
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate,1.208267
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate,0.953279
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High,0.945308
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High,-0.057187
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low,-0.226575


In [ ]:
import pandas as pd

performance = pd.read_csv("../data/raw/07_scheme_performance.csv")

risk = input("Enter Risk Appetite (Low / Moderate / High): ").strip().title()

result = (
    performance[
        performance["risk_grade"].str.contains(risk, case=False)
    ]
    .sort_values("sharpe_ratio", ascending=False)
    [
        [
            "scheme_name",
            "fund_house",
            "risk_grade",
            "sharpe_ratio",
            "return_3yr_pct",
            "expense_ratio_pct"
        ]
    ]
    .head(3)
)

print("\nTop 3 Recommended Funds\n")
print(result)

result.to_csv("../reports/fund_recommendations.csv", index=False)

print("\nRecommendation saved successfully.")

task 6

In [ ]:
import pandas as pd

holdings = pd.read_csv("../data/raw/09_portfolio_holdings.csv")
fund_master = pd.read_csv("../data/raw/01_fund_master.csv")

# Keep only Equity funds
equity_codes = fund_master[
    fund_master["category"] == "Equity"
]["amfi_code"]

equity_holdings = holdings[
    holdings["amfi_code"].isin(equity_codes)
].copy()

# Aggregate stock weights into sector weights
sector_weights = (
    equity_holdings.groupby(
        ["amfi_code", "sector"]
    )["weight_pct"]
    .sum()
    .reset_index()
)

# Convert percentage to decimal
sector_weights["weight"] = sector_weights["weight_pct"] / 100

# HHI = Σ(weight²)
hhi = (
    sector_weights.groupby("amfi_code")["weight"]
    .apply(lambda x: (x**2).sum())
    .reset_index(name="HHI")
)

hhi = hhi.merge(
    fund_master[
        ["amfi_code", "scheme_name", "fund_house"]
    ],
    on="amfi_code",
    how="left"
)

hhi = hhi.sort_values(
    "HHI",
    ascending=False
)

hhi.to_csv(
    "../reports/sector_hhi_report.csv",
    index=False
)

print(hhi.head(10))

# Advanced Insight 1

Funds with the most negative Historical VaR (95%) and CVaR exhibit the highest downside risk. Small-cap and sector-focused funds generally experienced larger tail losses than diversified large-cap funds.

# Advanced Insight 2

The Rolling 90-Day Sharpe Ratio shows that risk-adjusted performance changes over time. Large-cap funds maintained more stable Sharpe ratios, while mid-cap and small-cap funds experienced greater fluctuations during volatile market periods.

# Advanced Insight 3

Investor cohort analysis indicates that more recent investor cohorts contributed higher average SIP amounts. Equity-oriented funds remained the most preferred investment choice across multiple cohorts.

# Advanced Insight 4

SIP continuity analysis identified investors with an average SIP gap exceeding 35 days as at-risk. Maintaining consistent SIP contributions significantly improves long-term investment discipline.

# Advanced Insight 5

Sector concentration analysis using the Herfindahl-Hirschman Index (HHI) revealed that highly concentrated portfolios carry greater sector-specific risk, whereas diversified equity funds exhibit lower concentration and potentially better risk management.

In [ ]:
import os

files = [
    "../data/raw/07_scheme_performance.csv",
    "../reports/var_cvar_report.csv",
    "../reports/investor_cohort_analysis.csv",
    "../reports/sip_continuity_analysis.csv",
    "../reports/fund_recommendations.csv",
    "../reports/sector_hhi_report.csv"
]

print("Generated Files:\n")

for file in files:
    if os.path.exists(file):
        print("✅", file)
    else:
        print("❌ Missing:", file)